# LLaVA-OneVision: 轻松视觉任务转移
- [论文链接](https://arxiv.org/abs/2408.03326)

## 摘要

## 引言

借助大多模态模型（LMM）构建通用智能助手是人工智能研究的核心目标。LLaVA-OneVision 作为一款开源视觉 - 语言智能助手，延续了 LLaVA 系列的技术路线，通过轻量级连接模块将视觉编码器与大语言模型（LLM）相融合，遵循经济高效的研发范式。

LLaVA 系列已实现快速迭代升级：初代 LLaVA 展现出卓越的多模态对话能力；LLaVA-1.5 借助扩充学术指令数据，在多个基准测试中取得了当前最优（SoTA）性能；LLaVA-NeXT 则通过三项核心技术进一步突破性能边界，包括用于处理高分辨率图像的 AnyRes 技术、扩充高质量指令数据集、以及整合当时性能最优的开源大语言模型。作为具备可扩展性的技术原型，LLaVA-NeXT 催生了多项并行研究探索，相关成果均记录于其技术博客系列，涵盖零样本视频任务适配、大语言模型缩放效应、模型架构与训练策略的消融实验、以及多图像 / 视频 / 三维场景的能力拓展等方向。

上述研究均在固定算力预算内开展，旨在为项目推进提供有价值的参考洞见。研究团队同期在 1-6 月持续积累并整理了大规模高质量多模态数据集。LLaVA-OneVision 整合了这些研究洞见，并基于新积累的大规模数据集开展高效实验。该模型显著提升了开源大多模态模型在三大核心视觉场景（单图像、多图像、视频）的性能表现，同时实现跨场景任务迁移能力，例如基于图像任务迁移机制，大幅增强了视频理解能力。为推动通用视觉智能助手的研究发展，该项目已向公众开源多模态指令数据集、代码库、模型权重文件及可视化对话演示平台。

## 3 建模

## 3.1 网络架构

模型架构继承了 LLaVA 系列的极简设计，其主要目标在于($i$)有效利用 LLM 和视觉模型的预训练能力，以及 ($ii$) 在数据和模型方面实现强大的扩展性。网络架构如图 1 所示。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/overview.png" />
    <span style=" font-size: 12px; color: black;"><strong>图1</strong>：LLaVA-OneVision 网络架构。左：当前模型实例；右：[83]中 LLaVA 架构的一般形式，但扩展以支持更多视觉信号。</span>
</div>

1.  **模型组件选型**
    - **大语言模型（LLM）**：选用通义千问2（Qwen-2）作为参数为 $\boldsymbol{\phi}$ 的模型 $f_{\boldsymbol{\phi}}(\cdot)$，其支持多尺寸规格，在现有开源模型中具备领先的语言能力。
    - **视觉编码器**：采用SigLIP作为参数为 $\boldsymbol{\psi}$ 的视觉编码器 $g_{\boldsymbol{\psi}}(\cdot)$，可将输入图像 $\mathbf{X}_{\texttt{v}}$ 编码为视觉特征 $\mathbf{Z}_{\texttt{v}} = g(\mathbf{X}_{\texttt{v}})$。实验中分别采用Transformer最后一层前后的网格特征。
    - **投影层**：采用双层多层感知机（MLP）作为参数为 $\boldsymbol{\theta}$ 的投影层 $p_{\boldsymbol{\theta}}(\cdot)$，将图像特征映射至词嵌入空间，生成视觉token序列 $\mathbf{H}_{\texttt{v}} = p(\mathbf{Z}_{\texttt{v}})$。

2.  **选型依据**
    模型选型基于相关研究的实证结论：性能更强的LLM通常能显著提升模型在真实场景中的多模态能力；在开源视觉编码器中，SigLIP可使大多模态模型（LMM）取得更优性能。

3.  **目标答案概率计算**
    对于长度为 $L$ 的序列，目标答案 $\mathbf{X}_{\texttt{a}}$ 的概率计算方式如下：
    $$\begin{aligned}
    p( \mathbf{X}_{\texttt{a}} |  \mathbf{X}_{\texttt{v}}, \mathbf{X}_{\texttt{q}}) = \prod_{i=1}^{L} p (  {\color{mygreen} \mathbf{x}_i} | \mathbf{X}_{\texttt{v}}, \mathbf{X}_{\texttt{q}, <i}, \mathbf{X}_{\texttt{a}, <i})
    \end{aligned}$$
    其中 $\mathbf{X}_{\texttt{q}, <i}$ 和 $\mathbf{X}_{\texttt{a}, <i}$ 分别为当前预测token ${\color{mygreen} \mathbf{x}_i}$ 之前的所有轮次指令与答案token。公式中显式加入视觉信号 $\mathbf{X}_{\texttt{v}}$，强调其对所有答案生成的支撑作用。
    视觉信号 $\mathbf{X}_{\texttt{v}}$ 具备通用性，其输入形式随任务场景调整：单图像场景为图像切块，多图像场景为单张图像，视频场景则为单帧画面。

### 3.2 视觉表示

视觉信号的表征是视觉编码的核心，其效果取决于**原始像素空间分辨率**与**特征空间Token数量**两个要素，对应视觉输入表征配置为（分辨率，Token数）。

提升这两项指标均可优化模型性能，尤其在需要视觉细节的任务中效果显著。为平衡性能与计算成本，实验发现**提升分辨率比增加Token数更高效**，因此推荐采用带池化的AnyRes策略，相关对比见[图2](#higer_res)。

<a id="higer_res"></a>
<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/higher_res.png" />
    <span style=" font-size: 12px; color: black;"><strong>图2</strong>：视觉表示。顶部：采用双线性插值处理更高分辨率图像的新 Higher AnyRes 方案；底部：LLaVA-NeXT中的原始 AnyRes。</span>
</div>



__AnyRes策略原理__
对于宽高配置为$(a,b)$的AnyRes策略，图像会被划分为$a\times b$个切块，每个切块尺寸统一，适配视觉编码器输入要求。假设每个切块生成$T$个Token，结合1张基准缩览图，总Token数为$L=(a\times b+1)\times T$。
为控制Token总量，引入阈值$\tau$，若$L>\tau$则通过双线性插值降低单切块Token数，计算公式如下：
$$
T_{\text{new}} = 
\begin{cases} 
\frac{\tau}{(a \times b + 1)} & \text{if } L > \tau \\
T & \text{if } L \leq \tau 
\end{cases}
$$


<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/token_strategy.png" />
    <span style=" font-size: 12px; color: black;"><strong>图3</strong>：LLaVA-OneVision 中为每种场景分配 token 的视觉表示策略。不同场景中的视觉 token 数量被设计为相似，以确保平衡的视觉表示，以适应跨场景能力迁移。注意，729 是 SigLIP 编码分辨率为 384x384 的视觉输入所需的 token 数量。</span>
</div>

同时定义多组空间配置$(a,b)$以适配不同分辨率和长宽比的图像，实际选择切块数最少的配置。关于视觉表征的详细消融实验可参考文献[64]。

__跨场景表征策略__
本文提出的**高阶AnyRes策略**是一种灵活的视觉表征框架，可扩展至多图像与视频场景，其性能-成本最优配置可按需调整，具体细节见[图3]及章节C.1，核心策略如下：
- **单图像场景**：采用超大尺寸空间配置$(a,b)$，无需缩放即可保留原图分辨率；同时分配更多Token，构建长序列以充分表征视觉信息。此设计基于单图像高质量指令数据量远超视频数据的特点，通过模拟视频的长序列表征形式，实现从图像到视频理解的能力迁移。
- **多图像场景**：仅将基准分辨率图像输入编码器生成特征图，无需对高分辨率图像做多分块处理，节省计算资源。
- **视频场景**：将视频每一帧缩放到基准分辨率后输入编码器；通过双线性插值减少单帧Token数，从而在有限计算预算内处理更多帧，实现性能与成本的最优平衡。

上述表征配置均基于固定计算预算设计，旨在实现跨场景能力迁移。若算力充足，可在训练与推理阶段增加单张图像/单帧的Token数，进一步提升模型性能。

## 4 数据

==TODO==

## 5 训练策略

为赋予大语言模型（LLM）多模态能力，本文明确了三项核心功能，并将其系统划分为三个独立学习阶段，以支持消融实验。现有LLaVA模型的研究多聚焦于**单图指令微调**，而其他功能模块的探索较少，因此成为本节的核心研究重点。

模型训练遵循**课程学习（Curriculum Learning）原则**：按阶段依次训练难度递增的任务与样本。在计算资源固定的前提下，该策略可实现训练过程的解耦，并生成可复用的中间检查点，支持更多实验迭代。

1.  **阶段1：图文对齐（Language-Image Alignment）**
    - 核心目标：将视觉特征精准对齐至大语言模型的词嵌入空间。
    - 关键配置：仅训练视觉-语言投影层（projector）；基础图像表征对应729个视觉token；视觉编码器与投影层/LLM的学习率均为1×10⁻³。

2.  **阶段1.5：高质量知识学习（High-Quality Knowledge Learning）**
    - 核心目标：在计算效率与知识注入之间取得平衡，向多模态大模型（LMM）融入高质量知识。
    - 关键配置：训练配置与阶段2保持一致，确保知识无缝整合；启用AnyRes机制，视觉token数量最大提升至阶段1的5倍；全模型参与训练；视觉编码器学习率（2×10⁻⁶）为投影层/LLM（1×10⁻⁵）的1/5。

3.  **阶段2：视觉指令微调（Visual Instruction Tuning）**
    - 核心目标：让模型学会以理想响应完成多样化视觉任务，指令数据按类别分组训练。
    - 分两个子阶段：
      - **单图训练**：在320万单图指令数据上训练，使模型具备单图场景下的多任务指令跟随能力。
      - **OneVision训练**：在视频、单图、多图混合数据上训练，实现从单图到多场景的能力扩展，学习跨场景知识迁移，最终涌现新能力。该方法是当前赋予LMM多图与视频理解能力**最简且成本最优**的后训练方案。
    - 关键配置：AnyRes机制支持视觉token数量最大提升至阶段1的10倍；全模型训练，学习率配置与阶段1.5一致。

<!-- ### 训练策略核心总结 -->
<!-- 1.  **渐进式序列扩展**：训练过程中逐步提升最大图像分辨率与视觉token数量，实现长序列能力的渐进式培养。
2.  **模块训练策略**：阶段1仅训练投影层，后续阶段全模型训练，兼顾对齐效果与知识整合效率。
3.  **学习率差异化**：视觉编码器学习率始终为投影层与LLM的1/5，避免预训练视觉特征被过度破坏。
4.  **批量大小**：0.5B模型全局批量大小为512，7B与72B模型为256。
5.  **训练轮次**：所有阶段均训练1个epoch。 -->

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/training_configuration.png" />
    <span style=" font-size: 12px; color: black;"><strong>表1</strong>：LLaVA-OneVision 模型每个训练阶段的详细配置。该表概述了在课程学习过程的不同阶段中，视觉参数、数据集特性、模型规格和训练超参数的进展情况。我们对 0.5B 模型使用 512 的全局批处理大小，对 7B 和 72B 模型使用 256。</span>
</div>